# Matplotlib Intro — From Basics to a 3D Implicit Surface

This notebook is structured like a short class. Each section introduces a tool, gives you a small
practice problem to try yourself, then shows a walkthrough. By the end, you will have every piece
needed to plot the implicit surface from Question 1:

$$(x^2-1)^2(y^2-1)^2 + (y^2-1)^2(z^2-1)^2 + (z^2-1)^2(x^2-1)^2 = 0.08$$

**The strategy (stated up front):** build a 3D grid of candidate points, evaluate the equation's
left-hand side at every point, keep only the points where the result is close to 0.08 (within some
tolerance), and scatter-plot those surviving points. A denser grid gives a smoother surface; a
larger tolerance gives a thicker, more forgiving surface.

## Part 1: Matplotlib Basics

`matplotlib.pyplot` is the standard plotting interface. The core objects are:

- **Figure** — the whole window/canvas.
- **Axes** — a single plot living inside the figure (a figure can hold several axes/subplots).

The most common pattern is `fig, ax = plt.subplots()`, then calling methods on `ax` (`ax.plot`,
`ax.scatter`, `ax.set_xlabel`, ...). This is preferred over the older `plt.plot(...)` global style
because it stays explicit about which axes you're drawing on — important once you have multiple
subplots or a 3D axes object.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0, 2 * np.pi, 200)   # 200 evenly spaced points from 0 to 2*pi
y = np.sin(x)

fig, ax = plt.subplots()
ax.plot(x, y)
ax.set_xlabel("x")
ax.set_ylabel("sin(x)")
ax.set_title("A basic line plot")
plt.show()

**Key tool: `np.linspace(start, stop, num)`**

Returns `num` evenly spaced values between `start` and `stop` (inclusive). This is how you will
generate candidate x, y, z values later — it is the numeric backbone of the whole approach.

### Practice 1
Plot $y = x^2 - 1$ for $x$ ranging from $-3$ to $3$, using 100 points. Label the axes.

In [ ]:
# Try it yourself here


**Walkthrough**

In [ ]:
x = np.linspace(-3, 3, 100)
y = x**2 - 1

fig, ax = plt.subplots()
ax.plot(x, y)
ax.set_xlabel("x")
ax.set_ylabel("x^2 - 1")
plt.show()

## Part 2: Scatter Plots

`ax.scatter(x, y)` draws individual points instead of connecting them with a line. This matters for
our final approach: we are not plotting a clean function `y = f(x)`, we are plotting a *cloud* of
points that satisfy a condition, and those points have no natural line connecting them — a scatter
plot is the right tool, not a line plot.

### Practice 2
Generate 50 random points in $[-1, 1] \times [-1, 1]$ and scatter-plot only the ones that fall
inside the unit circle ($x^2 + y^2 \le 1$).

In [ ]:
# Try it yourself here


**Walkthrough**

In [ ]:
rng = np.random.default_rng(0)
pts = rng.uniform(-1, 1, size=(50, 2))
x, y = pts[:, 0], pts[:, 1]

mask = x**2 + y**2 <= 1     # boolean array: True where the condition holds

fig, ax = plt.subplots()
ax.scatter(x[mask], y[mask])
ax.set_aspect("equal")
plt.show()

**Key idea: boolean masking.** `mask` is an array of `True`/`False` values, one per point.
Indexing an array with a boolean array of the same shape (`x[mask]`) keeps only the entries where
the mask is `True`. This is exactly the mechanism used later to keep only points "close enough"
to the target value.

## Part 3: Grids with `np.meshgrid`

To evaluate a function over *every combination* of x and y values (not just paired-up points), use
`np.meshgrid`. Given 1D arrays of x-values and y-values, it returns 2D arrays `X` and `Y` such that
`(X[i,j], Y[i,j])` sweeps out every combination.

### Practice 3
Create `x = np.linspace(-2, 2, 5)` and `y = np.linspace(-2, 2, 5)`. Build the meshgrid, then compute
`Z = X**2 + Y**2` and print its shape.

In [ ]:
# Try it yourself here


**Walkthrough**

In [ ]:
x = np.linspace(-2, 2, 5)
y = np.linspace(-2, 2, 5)
X, Y = np.meshgrid(x, y)
Z = X**2 + Y**2

print(X.shape, Y.shape, Z.shape)   # all (5, 5): one Z value for every (x, y) pair

`meshgrid` is what turns two 1D lists of coordinates into the full 2D (or, with three inputs, 3D)
grid of points needed to evaluate a multivariable function everywhere at once, without writing a
manual nested loop.

## Part 4: 3D Plotting

3D plots need an axes created with `projection="3d"`. The two relevant methods are:

- `ax.plot_surface(X, Y, Z)` — for an explicit surface $z = f(x, y)$, using 2D meshgrid arrays.
- `ax.scatter(x, y, z)` — for plotting an arbitrary cloud of 3D points (1D arrays of equal length).

Our final problem is an *implicit* equation in x, y, z (not solvable as a clean $z = f(x,y)$), so
`scatter` on a filtered point cloud is the right tool, not `plot_surface`.

### Practice 4
Plot a 3D scatter of 200 random points inside the unit sphere ($x^2+y^2+z^2 \le 1$).

In [ ]:
# Try it yourself here


**Walkthrough**

In [ ]:
rng = np.random.default_rng(1)
pts = rng.uniform(-1, 1, size=(2000, 3))
x, y, z = pts[:, 0], pts[:, 1], pts[:, 2]

mask = x**2 + y**2 + z**2 <= 1

fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.scatter(x[mask], y[mask], z[mask], s=2)
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
plt.show()

## Part 5: Implicit Curves/Surfaces via Grid + Tolerance

An equation like $g(x, y, z) = c$ defines a surface implicitly. To visualize it with points on a
discrete grid, exact equality almost never happens (floating point, finite grid resolution). Instead:

1. Build a grid of candidate points (via `linspace` + `meshgrid`).
2. Evaluate $g$ at every grid point.
3. Keep points where $|g(\text{point}) - c| < \text{tol}$ for a small tolerance.
4. Scatter-plot the survivors.

A finer grid (more points) makes the surface look smoother and more continuous. A larger tolerance
keeps more points (thicker, more solid-looking surface but less precise); a smaller tolerance keeps
fewer points (thinner, sparser, more precise).

### Practice 5 (2D warm-up)
Use a grid to approximate the circle $x^2 + y^2 = 1$ as a scatter of points, using a tolerance of
0.05.

In [ ]:
# Try it yourself here


**Walkthrough**

In [ ]:
vals = np.linspace(-1.5, 1.5, 300)
X, Y = np.meshgrid(vals, vals)

G = X**2 + Y**2          # evaluate the left-hand side everywhere on the grid
tol = 0.05
mask = np.abs(G - 1) < tol

fig, ax = plt.subplots()
ax.scatter(X[mask], Y[mask], s=3)
ax.set_aspect("equal")
ax.set_title("Grid + tolerance approximation of x^2 + y^2 = 1")
plt.show()

This is the exact mechanism scaled up to 3D for the final problem — only the equation and the
number of dimensions change.

## Final Problem: The 3D Implicit Surface

$$(x^2-1)^2(y^2-1)^2 + (y^2-1)^2(z^2-1)^2 + (z^2-1)^2(x^2-1)^2 = 0.08$$

Same recipe as Practice 5, extended to three dimensions:

1. `linspace` for x, y, and z over a chosen range.
2. `np.meshgrid` (with three inputs) to build the full 3D grid.
3. Evaluate the left-hand side of the equation on the grid.
4. Mask points where the result is within `tol` of 0.08.
5. `ax.scatter` the survivors in 3D.

Try setting up the grid and mask yourself before looking at the walkthrough.

In [ ]:
# Try it yourself here


**Walkthrough**

In [ ]:
n = 80                      # points per axis -- raise this for a smoother surface
vals = np.linspace(-1.6, 1.6, n)
X, Y, Z = np.meshgrid(vals, vals, vals)

lhs = (X**2 - 1)**2 * (Y**2 - 1)**2 \
    + (Y**2 - 1)**2 * (Z**2 - 1)**2 \
    + (Z**2 - 1)**2 * (X**2 - 1)**2

tol = 0.01                  # raise for a thicker/more solid-looking surface
mask = np.abs(lhs - 0.08) < tol

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(projection="3d")
ax.scatter(X[mask], Y[mask], Z[mask], s=2, alpha=0.5)
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.set_title("(x^2-1)^2(y^2-1)^2 + (y^2-1)^2(z^2-1)^2 + (z^2-1)^2(x^2-1)^2 = 0.08")
plt.show()

### Notes on tuning

- **Grid density (`n`)**: memory and runtime scale as $n^3$ (a 3D grid), so going from 80 to 200 per
  axis is an 8x cost increase, not 2.5x. Start small while testing, then raise `n` for the final render.
- **Tolerance (`tol`)**: too small and few/no points survive on a coarse grid; too large and the
  surface looks thick or picks up unwanted regions. Tune tolerance and grid density together —
  a finer grid can tolerate a smaller tolerance.
- **Range of values**: the range in `linspace` should be wide enough to contain the whole surface;
  if the scatter looks cut off at the edges, widen it.